# **Measure Regions Morphology**

# <mark> TO DO LIST:
- <mark> check how volume fraction is handled when mask_name=None (morph base functions, 2.1, 2.2, and 2.3)
- <mark> update 2.1 to summarize organelle intensity value in summary stats function (currently left out)
- <mark> add one object per region check step (included in get_regions_morpho function) to other get_*() functions in 2.1, 2.2, and 2.3
- <mark> double check that get_morphology_metrics() can take channel_axis=0 when mask_name=None
- <mark> Need to fix "mask_vol = regionprops_table(mask,properties=["area"], spacing=scale)['area'][0]" in get_morphology_metrics, so it can run with mask_name=None/mask=None
- <mark> update 2.1 first function steps to have mask and intensity defining included (updating step naming to match here)

***Prior to this notebook, you should have already run through [2.0_quantification_setup](2.0_quantification_setup.ipynb) and have segmentation outputs of <ins>at least one</ins> region or mask (e.g., cell mask, nucleus, neurites, etc.).***

In notebooks 2.1 through 2.4, we will go over the implementation of `infer-subc` quantification methods (explained in detail in the `method_...` notebooks) to assess the morphology, interactions, and distribution of organelles at the single-cell level. 

### 📍 **Purpose**
This notebook can be used to measure the `morphology` -- the amount, size, and shape -- of one or more masks/regions within images. It includes options to:
1. 🦠 Quantify the morphology of *one or more mask(s)/region(s)* from <ins>ONE IMAGE</ins>
2. 🧪 Batch process the morphology of *one or more mask(s)/region(s)* from *multiple images* for a <ins>SINGLE EXPERIMENT</ins>
3. 🧮 Summarize morphology metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>

Below you will find an `explanation of steps` for these three quantitative methods. Then, more succinct code blocks are included to `execute quantification` on your own (or sample) data.

### 🍃 **Biological Relevance - Mask/region Morphology**
Outside of organelle-specific metrics (morphology, interactions, distribution), it can also be beneficial to understand more macro-level information about biological relevant sub-regions within your images. Some examples of this may be the whole or subcellular regions such as the nucleus/cytoplasm or neurites/soma.

In this analysis notebook, the morphology of masks/regions are measured using the methods described in [method_morphology.ipynb](method_morphology.ipynb). The following morphological measurements are included for each organelle:
- `label`: the unique ID number for the object being measured
- `centroid`: centroid coordinate tuple (row, col, Z)
- `bbox`: bounding box coordinates (min_row, min_col, max_row, max_col); pixels/voxels belonging to the bounding box are in the half-open interval [min_row; max_row) and [min_col; max_col).
- `area`: (or `volume` for 3D z-stack images) area of the region i.e. number of pixels of the region scaled by pixel-area; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `surface_area`: the surface area of the region. For 3D, surface area of a 2D surface mesh of the region (skimage.measure.marching_cubes) using skimage.measure.mesh_surface_area; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `SA_to_volume`: surface area / area (or volume); this metric has the option to be converted into "real world" units using the scale from the metadata.
- `equivalent_diameter`: the diameter of a circle with the same area as the region; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `extent`: ratio of pixels/voxels in the region to pixels/voxels in the total bounding box. Computed as area / (rows * cols)
- `euler_number`: Euler characteristic of the set of non-zero pixels. Computed as number of connected components subtracted by number of holes (input.ndim connectivity). In 3D, number of connected components plus number of holes subtracted by number of tunnels.
- `solidity`: ratio of pixels/voxels in the region to pixels/voxels of the convex hull image.
- `axis_major_length`: the length of the major axis of the ellipse that has the same normalized second central moments as the region; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `mask_volume`: the volume of the mask used to define the area of analysis; usually this will be the cell mask since the analysis is intended to be done at the single-cell level.

The following measures of the intensity images are also included:
- `min_intensity`: value with the least intensity in the region.
- `max_intensity`: value with the greatest intensity in the region.
- `mean_intensity`: value with the mean intensity in the region.
- `standard_deviation_intensity`: the standard deviation of the intensity in the region.


These measurements and definitions are derived from the [`skimage.measure.regionprops()`](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops) function. More in depth information about each measurement can be found there.

*You can learn more about the implementation of regionprops within infer-subc in the [method_morphology](method_morphology.ipynb) notebook.*

-----

## 🗂️ **Table of Contents**
The following sections are included in this notebook:

**IMPORTS AND LOAD IMAGE**

**EXPLANATION OF STEPS** - This section serves as *expository examples* of the functions used to quantify, batch process, and summarize organelle morphology.

🦠 **Quantify the morphology of *one or more regions/masks* from <ins>ONE IMAGE</ins>**
- **`STEP 1`** - Define the cell mask & include intensity if desired
- **`STEP 2`** - Loop through the list of regions to quantify their morphology metrics
- **`STEP 3`** - Combine all of the tables together and add the image name as a metadata column
- **`DEFINE`** - The get_regions_morphology() function

🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 1`** - Check the output file paths to see if quantification data already exists
- **`STEP 2`** - List images and the segmentation file suffixes that should be collected for each
- **`STEP 3`** - Loop through the list of images and perform the morphology quantification on all organelles
- **`DEFINE`** - The batch_process_regions_morph() function

🧮 **Summarize metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>**
- **`STEP 1`** - Get the regions morphology .csv files
- **`STEP 2`** - Summarize the mean, median, and standard deviation of each metric per image/mask region
- **`STEP 3`** - Ensure all regions included in the analysis are represented and fill NA values with 0 as needed
- **`STEP 4`** - Unstack the region names and save file
- **`DEFINE`** - The batch_org_morph_summary_stats() function

**EXECUTE QUANTIFICATION** - Once you understand how the functions work, this section can be used to quantify your data in a quick and easy way.
- **`STEP 1`:** 🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 2`:** 🧮 **Summarize metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>**

-----
---------------------
## **IMPORTS AND LOAD IMAGE**
Details about the functions included in this subsection are outlined in the [`2.0_quantification_setup`](2.0_quantification_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

In [ ]:
from typing import List, Union
from pathlib import Path
import os
import time
import warnings
import tempfile

from infer_subc.core.img import *

import numpy as np
import pandas as pd
import napari
from napari.utils.notebook_display import nbscreenshot

from infer_subc.quantification.morphology import get_morphology_metrics
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image
from infer_subc.quantification.batch import load_existing_keys_csv, append_atomic_csv
from infer_subc.quantification.regions import batch_regions_morph_summary_stats, batch_process_regions_morph

pd.set_option('display.max_columns', None)

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information: `sample_data_type`, `data_root_path`, `raw_data_path`, `seg_data_path`, `quant_data_path`, `raw_img_type`, and `seg_img_type`.

In [ ]:
### USER INPUT REQUIRED ###
# If using the sample data, select which cell type you would like analyze:
sample_data_type = "pri-neuron"


# If you are not using the sample data, please edit "USER SPECIFIED" as necessary.
data_root_path = Path("USER SPECIFIED")

raw_data_path = data_root_path / "USER SPECIFIED"

seg_data_path = data_root_path / "USER SPECIFIED"

quant_data_path = data_root_path / "USER SPECIFIED"

raw_img_type = "USER SPECIFIED"

seg_img_type = "USER SPECIFIED"

#### &#x1F3C3; **Run code; no user input required**

Specify the sample data information (if using), create the output path if it doesn't exist, and print the list of files in the input.

In [ ]:
# If sample_data_type is set to "pri-neuron", "pri-astrocyte", "ineuron" or "iPSC" then the sample data is used and the directories are set
if sample_data_type != None:
    data_root_path, raw_img_type, seg_img_type, raw_data_path, seg_data_path, quant_data_path = sample_input_quant(sample_data_type)

# Create the output directory to save the segmentation outputs in.
if not Path.exists(quant_data_path):
    Path.mkdir(quant_data_path)
    print(f"making {quant_data_path}")

# Create a list of the file paths for each image in the input folder. Select test image path.
raw_img_file_list = list_image_files(raw_data_path,raw_img_type)
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":raw_img_file_list})

#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: `test_img_n`

In [ ]:
#### USER INPUT REQUIRED ###
test_img_n = 0

#### &#x1F3C3; **Run code; no user input required**

Read in the image and metadata; visualize the image in Napari.

In [ ]:
# Read in the image and metadata as an ndarray and dictionary from the test image selected above. 
test_img_name = raw_img_file_list[test_img_n]
img_data,meta_dict = read_czi_image(test_img_name)

# Define some of the metadata features.
channel_names = meta_dict['name']
meta = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']
file_path = meta_dict['file_name']

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale (ZYX): {scale}")
print(f"Channel axis: {channel_axis}")

#### &#x1F6D1; &#x270D; **User Input Required:**

Specify the following information about the segmentation files: - `org_file_names`, `org_channels_ordered`, `regions_file_names`, `suffix_separator`, and `mask_name`.

In [ ]:
#### USER INPUT REQUIRED ###
org_file_names = "USER SPECIFIED"

org_channels_ordered = "USER SPECIFIED"

regions_file_names = "USER SPECIFIED"

mask_name = "USER SPECIFIED"

suffix_separator = "USER SPECIFIED"

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
# specify the organelle names, the order of the intensity channels that match the organelle segmentations, and the region names for the sample data or your own data.
if sample_data_type != None:
    org_file_names, org_channels_ordered, regions_file_names, mask_name, suffix_separator = sample_quant_settings(sample_data_type)


# find file paths for segmentations
all_suffixes = org_file_names + regions_file_names
filez = find_segmentation_tiff_files(file_path, all_suffixes, seg_data_path, suffix_separator)

# print paths to matching seg files
print("The following matching files were found:")
for k, i in filez.items():
    print(f"{k}: {i}")

# read the segmentation and masks/regions files into memory
organelles = [read_tiff_image(filez[org]) for org in org_file_names]
regions = [] 
for m in regions_file_names:
    mfile = read_tiff_image(filez[m])
    regions.append(mfile)

# match the intensity channels to the segmentation files
intensities = [img_data[ch] for ch in org_channels_ordered]

# specifiy the mask image
m = regions_file_names.index(mask_name)
mask = regions[m]


# open viewer and add images
viewer = napari.Viewer()
for r, reg in enumerate(regions_file_names):
    viewer.add_image(regions[r],
                     scale=scale,
                     name=f"{reg} mask")

# colors = ["red", "bop orange", "yellow", "green", "blue", "cyan", "magenta", "bop purple"]
for o, org in enumerate(org_file_names):
    viewer.add_image(intensities[o],
                     scale=scale,
                     name=f"{org} intensity channel")
    viewer.add_labels(organelles[o],
                      scale=scale,
                      name=f"{org} segmentation")
viewer.grid.enabled = True
viewer.reset_view()
    
print("Proceed to Napari window to view your selected image.")

# screenshot viewer
nbscreenshot(viewer, canvas_only = False)

------
-----
## **EXPLANATION OF STEPS** <a id='explanation'></a>

-----
### 🦠 **Quantify one or more regions/masks from <ins>ONE IMAGE</ins>**

#### **`STEP 1` - Define the cell mask & include intensity if desired**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** A single region included in the analysis can also be defined as the mask, the area in which the other regions must be within to be analyzed. Any regions outside of the area defined by the mask will be excluded. If no mask if specified, then this rule will not apply and anything within the image will be quantified. In the analysis below, we will walk through an example where a single cell and single nucleus are included in the regions analysis. The cell is specified as the mask, limiting the analysis to within the cell area only. The other regions being analyzed, the nucleus, will need to fall within the cell mask to be quantified. 

In addition to define the mask object, the cell block below also specifies the intensity image to include in the morphology analysis, if intensity quantification is desired. Any number of channels can be included in the intensity analysis here. 

*Note: the inputs specified above will be used here (e.g., `mask_name` and `intensities`).*

In [ ]:
# specify the mask image to use during quantification
if mask_name is None:
    mask = None
    print("No mask name provided. No mask will be applied before analysis.")
elif mask_name not in list_region_names:
    print(f"Mask '{mask_name}' not found in `list_region_names`:{list_region_names}. No mask will be applied before analysis.")
    mask = None
    mask_name = None
else:
    mask = list_region_segs[list_region_names.index(mask_name)]
    print(f"Using '{mask_name}' as the mask for analysis.")


# merge intensity images to create a single np.ndarray
if intensities is None:
    intensity_img = None
    print("No intensity images provided. Morphology metrics that require intensity images will not be calculated.")
else:
    intensity_img = np.stack(intensities, axis=0)
    print(f"Including {len(intensities)} intensity channels for analysis.")

#### **`STEP 2` - Loop through the list of regions to quantify their morphology metrics**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The code block below loops through the list of region names, finds the associated segmentation from the region segmentation list, ensures all segmentations have only one object, and then quantifies the regions' morphology metrics using `get_morphology_metrics()` from [`method_morphology.ipynb`](method_morphology.ipynb). 

A constraint of the current infer-subc analysis pipeline is that only a single region or mask area can be analyzes per image for each mask/region type. For example, only a single cell or a single nucleus can be analyzed per image. ***If more than one mask/region are included in the segmentation image, the regions will be combined before analysis** as shown in the code block below.*

In [ ]:
# empty list to collect a morphology data for each organelle
regions_tab = []

# loop through the list of organelles and run the get_morphology_metrics function
for j, target in enumerate(regions_file_names):
    region_seg = regions[j]

    # verify only one object per mask image; if more than one, combine them into a single object
    ## TODO: update to multi-object analysis later
    unique_objs = np.unique(region_seg)
    unique_objs = unique_objs[unique_objs != 0]  # exclude background
    if len(unique_objs) > 1:
        warnings.warn(f"More than one object found in region segmentation '{regions_file_names[j]}'. Combining all objects into a single object for analysis.")
        region_seg = (region_seg > 0).astype(int)

    # run get_morphology_metrics function to output a table of measurements
    region_metrics = get_morphology_metrics(segmentation_img=region_seg, 
                                            seg_name=target,
                                            intensity_img=intensity_img, 
                                            intensity_ch_names=org_file_names,
                                            channel_axis=0, # default to 0 because intensities are merged from list on axis 0
                                            mask=mask,
                                            mask_name=mask_name,
                                            scale=scale)
    
    # add table to list above
    regions_tab.append(region_metrics)

#### **`STEP 3` - Combine all of the tables together and add the image name as a metadata column**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block combines the above tables together so that each organelle object is listed as a separate column in a single table. A new column is then added to specify which image the data is from.

In [ ]:
# combine the lists for each organelle into one table
final_region_tab = pd.concat(regions_tab, ignore_index=True)

# add a new column to list the name of the image these data are derived from 
final_region_tab.insert(loc=0,column='image_name',value=file_path.stem)

print("Final morphology metrics table for all regions:")
display(final_region_tab)

#### **`DEFINE` - The get_regions_morphology() function**

In [ ]:
def _get_regions_morphology(source_file_path: str,
                           list_region_names: Union[List[str], None]=None,
                           list_region_segs: Union[List[np.ndarray], None]=None,
                           list_intensity_img: Union[List[np.ndarray], None]=None,
                           list_channel_names: Union[List[str], None]=None,
                           mask_name: Union[str, None]=None,
                           scale: Union[tuple, None]=None) -> pd.DataFrame:
    """
    Measure morphology metrics of masks/regions included in the large infer-subc pipeline (e.g. cell, nucleus, etc.).

    Parameters
    ------------
    source_file: str
        Path to the source image file. This will be used as part of the metadata information in the output table. 
        The input images are not derived from this path, but rather are provided directly as arrays in the list_obj_segs and 
        list_intensity_img variables below.
    list_region_names: Union[List[str], None]
        List of segmented region/mask names. These names should match the suffix on the segmentation image files.
        This should include:
            - a mask segmentation, such as the cell mask, for masking during all interactions analysis; else, the entire image will be 
            quantified. Only one objects per mask image will be analyzed. If there are more than one included, they will be combined 
            prior to analysis and the entire region will be quantified. If no mask is provided, the entire image will be quantified.
            - a centering object, such as the nucleus, for distribution analysis; else the center of the mask region will be used as 
            the XY distribution centering point if distribution analysis is included.
    list_region_segs: Union[List[np.ndarray], None]
        List of 3D region segmentation arrays matching the order specified in list_region_names. Specify None if no regions are provided.
    list_intensity_img: Union[List[np.ndarray], None]
        List of 3D intensity channels from the raw image. Any number of channels can be included.
        These names will be used to rename the intensity measurement columns in the output table.
        If no intensity analysis is to be included, specify None here.
    list_channel_names: Union[List[str], None]
        List of names for each intensity channel provided in list_intensity_img. The order should match the order of the channels in list_intensity_img.
    mask_name: Union[str, None]
        Name of the region to use as the mask for analysis; if not specified, the entire image will be quantified.
        The mask_name should match one of the names provided in list_region_names. This object will be used to mask
        all other objects before quantitative analysis is performed. It will also be included as one of the analyzed objects.
    scale: Union[tuple,None] = None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
            
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each object in the segmentation image (rows) and the regionprops object

    """
    # Validate inputs
    if list_region_names is None or list_region_segs is None:
        raise ValueError("You must provide both list_region_names and list_region_segs arguments.")
    if len(list_region_names) != len(list_region_segs):
        raise ValueError("The length of list_region_names must match the length of list_region_segs.")
    if list_intensity_img is not None and list_channel_names is None:
        raise ValueError("You must provide both list_intensity_img and list_channel_names arguments to quantify intensity metrics.")
    if list_intensity_img is not None and list_channel_names is not None and len(list_intensity_img) != len(list_channel_names):
        raise ValueError("The length of list_intensity_img must match the length of list_channel_names.")

    if isinstance(source_file_path, str): source_file_path = Path(source_file_path)
    print(f"Quantifying region morphology from {source_file_path.name}")


    # specify the mask image to use during quantification
    if mask_name is None:
        mask = None
        print("No mask name provided. No mask will be applied before analysis.")
    elif mask_name not in list_region_names:
        print(f"Mask '{mask_name}' not found in `list_region_names`:{list_region_names}. No mask will be applied before analysis.")
        mask = None
        mask_name = None
    else:
        mask = list_region_segs[list_region_names.index(mask_name)]
        print(f"Using '{mask_name}' as the mask for analysis.")

    # merge intensity images to create a single np.ndarray
    if list_intensity_img is None:
        intensity_img = None
        print("No intensity images provided. Morphology metrics that require intensity images will not be calculated.")
    else:
        intensity_img = np.stack(list_intensity_img, axis=0)

    # empty list to collect a morphology data for each organelle
    regions_tab = []

    # loop through the list of organelles and run the get_morphology_metrics function
    for j, target in enumerate(list_region_names):
        region_seg = list_region_segs[j]

        # verify only one object per mask image; if more than one, combine them into a single object
        ## TODO: update to multi-object analysis later
        unique_objs = np.unique(region_seg)
        unique_objs = unique_objs[unique_objs != 0]  # exclude background
        if len(unique_objs) > 1:
            warnings.warn(f"More than one object found in region segmentation '{list_region_names[j]}'. Combining all objects into a single object for analysis.")
            region_seg = (region_seg > 0).astype(int)
        else:
            region_seg = region_seg

        # run get_morphology_metrics function to output a table of measurements
        region_metrics = get_morphology_metrics(segmentation_img=region_seg, 
                                            seg_name=target,
                                            intensity_img=intensity_img, 
                                            intensity_ch_names=list_channel_names,
                                            channel_axis=0, # default to 0 because intensities are merged from list on axis 0
                                            mask=mask,
                                            mask_name=mask_name,
                                            scale=scale)
        
        # add table to list above
        regions_tab.append(region_metrics)

    # combine the lists for each organelle into one table
    final_region_tab = pd.concat(regions_tab, ignore_index=True)

    # add a new column to list the name of the image these data are derived from 
    final_region_tab.insert(loc=0,column='image_name',value=source_file_path.stem)

    return final_region_tab

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [ ]:
# test function above
region_morph_tab = _get_regions_morphology(source_file_path = test_img_name,
                                          list_region_names = regions_file_names,
                                          list_region_segs = regions,
                                          list_intensity_img = intensities,
                                          list_channel_names = org_file_names,
                                          mask_name=mask_name,
                                          scale=scale)
display(region_morph_tab)

# check is function output matches previous output
print(f"\nThe output here matches the output from the individual steps above: {region_morph_tab.equals(final_region_tab)}")

##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.regions` and can be imported with the following:
> ```python
> from infer_subc.quantification.regions import get_regions_morphology
> ```

In [ ]:
from infer_subc.quantification.regions import get_regions_morphology

# test function above
region_morph_final = get_regions_morphology(source_file_path = test_img_name,
                                          list_region_names = regions_file_names,
                                          list_region_segs = regions,
                                          list_intensity_img = intensities,
                                          list_channel_names = org_file_names,
                                          mask_name=mask_name,
                                          scale=scale)
display(region_morph_final)

# check is function output matches previous output
print(f"\nThe output here matches the output from the individual steps above: {region_morph_tab.equals(region_morph_final)}")

-----
### 🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**

#### **`STEP 1` - Check the output file paths to see if quantification data already exists**

##### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** The quantification pipeline outlined here will write new quantitative data to a csv file as each image is processed. This ensures that data is not lost if processing is interrupted. This step first checks if an output file of the same name already exists and lists any data included there. 

Specify the name of this dataset:
- `dataset_name`: A unique string identifier for this dataset. It will be included as metadata in output tables and as part of the output files names. It will be used to identify if any data has already been collected for this dataset.

In [ ]:
### USER INPUT REQUIRED ###
dataset_name = "test_dataset_1"

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This step defines the output csv file paths that will be created during interaction analysis batch processing below and checks if any data from this dataset already exists there. If so, the images that have quantitative data in each file will not be processed again. 

In [ ]:
# define a function to check for existing data in csv files
def _load_existing_keys_csv(csv_path, key_cols, chunksize=250_000):
    keys = set()
    if not os.path.exists(csv_path):
        return keys
    for chunk in pd.read_csv(csv_path, usecols=key_cols, chunksize=chunksize):
        keys.update(map(tuple, chunk[key_cols].itertuples(index=False, name=None)))
    return keys


# specify the columns that will be checked per table
unique_keys = ['dataset', 'image_name']

# check if any existing data is present in outfiles
regions_path = quant_data_path / f"{dataset_name}_regions_morphology_metrics.csv"
existing_morpho_keys = _load_existing_keys_csv(regions_path, unique_keys)

# show data that was found
if existing_morpho_keys:
    print(f"Found existing {dataset_name} data for {len(existing_morpho_keys)} images in {quant_data_path}. These images will be skipped during analysis to avoid duplicate data entries.")
else:
    print(f"No \"{dataset_name}\" data was found in {quant_data_path}. All images will be processed during analysis.")

#### **`STEP 2` - List images and the segmentation file suffixes that should be collected for each**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** These steps collect a list of the images included in your "raw" (intensity image) data folder. Then, the masks suffixes and organelle suffixes are combined into one list.

In [ ]:
# reading list of files from the raw path
img_file_list = list_image_files(raw_data_path, raw_img_type)
print(f"Found {len(img_file_list)} images in {raw_data_path} for analysis:")
display(pd.DataFrame({"Image Name":img_file_list}))

print(f"The following segmentation files will be collected for each image: {regions_file_names}")

#### **`STEP 3` - Loop through the list of images and perform the morphology quantification on all regions**

##### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** This step loops through each image listed, reads the segmentation files, formats them appropriately, and then carries out morphology quantification via `get_regions_morphology()`.

Specify if the quantification should be carried out with or without the scale:
- `use_scale`: True indicates that the function will use the scale metadata to produce "real world" metrics (e.g., microns, etc.). False will produce quantification results in pixel/voxel units.

*Note: the file path locations and file information provided in the beginning of the notebook will be used here for simplicity.*

In [ ]:
### USER INPUT REQUIRED ###
use_scale = True

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below loops through the list of files and runs the `get_regions_morphology()` function on each one. The loop utilizes the following sequence of steps:
1) Find the paths for all of the organelle and mask segmentation files.
2) Collect the intensity channels and organelle segmentation files in the same order. Store them as lists.
3) Collect all of the region segmentation files in another list.
4) Determine the scale from the metadata if it is being used.
5) Run the get_regions_morphology() function and add the resulting data table to the org_tab list.
6) Repeat the loop above for each image in the raw image list and add them sequentially to the org_tab list.

In [ ]:
# define function to append data to csv after each image is processed
def _append_atomic_csv(csv_path, df):
    if df.empty:
        return
    if not os.path.exists(csv_path):
        df.to_csv(csv_path, index=False)
        return
    existing = pd.read_csv(csv_path)
    merged = pd.concat([existing, df], axis=0, ignore_index=True)
    merged = merged.drop_duplicates()  # final guard against repeat values
    fd, tmp_path = tempfile.mkstemp(prefix='append_', suffix='.csv')
    os.close(fd)
    merged.to_csv(tmp_path, index=False)
    os.replace(tmp_path, csv_path)


# loop through list of images and quantify interaction metrics if data for that image do not already exist;
for img_f in img_file_list:
    # skip files that have already been processed
    if (dataset_name, img_f.stem) in existing_morpho_keys:
        print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
        continue

    # process analysis for this cells
    else:
        filez = find_segmentation_tiff_files(img_f, regions_file_names, seg_data_path, suffix_separator)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # create intensities from raw file as list based on the channel order provided
        if org_file_names is None:
            intensities = None
            print("No intensity channel information provided.")
        else:
            if channel_axis != 0:
                img_data = np.moveaxis(img_data, channel_axis, 0)
            intensities = [img_data[i] for i, ch in enumerate(org_file_names) if ch is not None]
            channel_names = [ch for ch in org_file_names if ch is not None]

        # store region images as list
        regions = [read_tiff_image(filez[r]) for r in regions_file_names] 

        # define the scale
        if use_scale is True:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None

        regions_metrics = _get_regions_morphology(source_file_path=img_f,
                                                  list_region_names=regions_file_names,
                                                  list_region_segs=regions,
                                                  list_intensity_img=intensities,
                                                  list_channel_names=channel_names,
                                                  mask_name=mask_name,
                                                  scale=scale_tup)

        # save the morphology (or labels only) table data per image directly to csv
        regions_metrics.insert(loc=0,column='dataset',value=dataset_name)
        _append_atomic_csv(regions_path, regions_metrics)
        del regions_metrics  # free up memory

#### **`DEFINE` - The batch_process_regions_morph() function**

The following code combines the steps above into a single function that can be accessed as part of the `infer-subc` package. The function takes in file path information for intensity and segmentation images and the parameters necessary to quantify regions morphology metrics. 

This function can be utilized from infer-subc using:
```python
infer_subc.quantification.regions.batch_process_regions_morph()
```

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block defines a prototypes of the `batch_process_regions_morph()` function based on the steps above and tests it below.

In [ ]:
def _batch_process_regions_morph(dataset_name: str,
                                 raw_path: Union[Path,str], 
                                 seg_path: Union[Path,str],
                                 quant_path: Union[Path, str], 
                                 raw_file_type: str,
                                 region_names: Union[List[str], None],
                                 channel_axis: Union[int, None]=None,
                                 channel_names: Union[List[int], None]=None,
                                 mask_name: Union[str, None]=None,
                                 use_scale: bool=True,
                                 seg_suffix: Union[str, None]=None):
    """  
    batch process quantification of the regions morphology for a single dataset. This function is currently 
    optimized to process images from one file folder per image type (e.g., raw, segmentation) the output csv 
    files are saved to the indicated quant_path folder.

    Parameters:
    ----------
    dataset_name : str
        A unique string identifier for the dataset being processed. It will be included as metadata in output tables and as 
        part of the output files names. It will be used to identify if any data has already been collected for this dataset.
    raw_path: Union[Path,str]
        Path or str to the folder that contains the raw image files
    seg_path: Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files
    quant_path: Union[Path, str]
        Path or str to the folder that the output datatables will be saved to
    raw_file_type: str
        File type of the raw images (e.g., "czi", "tiff")
    region_names: Union[List[str], None]=None
        List of region names to analyze. Usually ['cell', 'nuc'] for cell mask and nucleus.
        If no regions are to be included, specify None here.
    channel_axis : int
        Axis corresponding to the channels in the image data
    channel_names: List[str]
        List of channel names associated to each channel in the raw image data; if you wish to exclude a particular channel
        from the intensity analysis, write None instead of the channel name.
    mask_name: Union[str, None]=None
        Name of the region to use for segmentation (if any). This name should be included in the regions_name variable.
        If None, the entire image will be quantified.
    use_scale: bool=True
        Whether to apply scaling to the quantitative data; scaled data will be in real world units (e.g., microns) rather than pixels/voxels
    seg_suffix:Union[str, None]=None
        Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix, not including the initial "-"

    Returns:
    ----------
    None: files are saved to quant_path directly
    """
    
    start = time.time()
    count = 0

    # create path objects if inputs are strings
    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(quant_path, str): quant_path = Path(quant_path)
    
    # create directory is it doesn't exist
    if not Path.exists(quant_path):
        Path.mkdir(quant_path)
        print(f"Output file path not found. Making {quant_path}.")

    if region_names is None:
        raise ValueError("No region names provided. Please provide at least one region name to analyze.")

    # check if any existing data is present in outfiles to skip already processed images
    unique_keys = ['dataset', 'image_name']

    regions_path = quant_path / f"{dataset_name}_regions_morphology_metrics.csv"
    existing_morpho_keys = load_existing_keys_csv(regions_path, unique_keys)

    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)
    len_file_list = len(img_file_list)

    # loop through list of cell analyzing each and appending the data to the empty list
    for img_f in img_file_list:
        img_start = time.time()
        count = count + 1
        # skip files that have already been processed
        if (dataset_name, img_f.stem) in existing_morpho_keys:
            print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
            continue
        # process analysis for this cells
        else:
            filez = find_segmentation_tiff_files(img_f, region_names, seg_path, seg_suffix)

            # read in raw file and metadata
            img_data, meta_dict = read_czi_image(filez["raw"])

            # create intensities from raw file as list based on channel_name list
            if channel_names is None:
                intensities = None
                print("No intensity channel information provided.")
            else:
                if channel_axis != 0:
                    img_data = np.moveaxis(img_data, channel_axis, 0)
                intensities = [img_data[i] for i, ch in enumerate(channel_names) if ch is not None]
                channel_names = [ch for ch in channel_names if ch is not None]

            # store region images as list
            regions = [read_tiff_image(filez[org]) for org in region_names]

            # define the scale
            if use_scale is True:
                scale = meta_dict['scale']
            else:
                scale = None

            regions_metrics = _get_regions_morphology(source_file_path=img_f,
                                                  list_region_names=region_names,
                                                  list_region_segs=regions,
                                                  list_intensity_img=intensities,
                                                  list_channel_names=channel_names,
                                                  mask_name=mask_name,
                                                  scale=scale)
            
            # save the morphology table data per image directly to csv
            regions_metrics.insert(loc=0,column='dataset',value=dataset_name)
            append_atomic_csv(regions_path, regions_metrics)
            del regions_metrics  # free up memory

            end2 = time.time()
            print(f"Completed quantification of {meta_dict['file_name']} in {(end2-img_start)/60} mins.")
            print(f"{count}/{len_file_list} images have been processed.")
            print(f"Time elapsed: {(end2-img_start)/60} mins")

    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{quant_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [ ]:
# define a new dataset name for comparison
dataset_name2 = "test_dataset_1-comparison"

_batch_process_regions_morph(dataset_name=dataset_name2,
                        seg_path = seg_data_path,
                        quant_path = quant_data_path, 
                        raw_path = raw_data_path, 
                        raw_file_type = raw_img_type,
                        channel_axis = channel_axis,
                        channel_names = org_file_names,
                        region_names = regions_file_names,
                        mask_name = mask_name,
                        use_scale = use_scale,
                        seg_suffix = suffix_separator)

# compare the output files from both dataset runs to ensure they are identical
morpho_file1 = quant_data_path / f"{dataset_name}_regions_morphology_metrics.csv"
morpho_file2 = quant_data_path / f"{dataset_name2}_regions_morphology_metrics.csv"


# determine if files are identical except for the dataset name column
if morpho_file1.exists() and morpho_file2.exists():
    morph_1 = pd.read_csv(morpho_file1)
    morph_2 = pd.read_csv(morpho_file2)
    morph_2['dataset'] = morph_1['dataset']  # set dataset names to be the same for comparison
    print("Regions metrics files identical:", morph_1.equals(morph_2))

##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.regions` and can be imported with the following:
> ```python
> from infer_subc.quantification.regions import batch_process_regions_morph
> ```

In [ ]:
from infer_subc.quantification.regions import batch_process_regions_morph

# define a new dataset name for comparison
dataset_name3 = "test_dataset_1-final"

batch_process_regions_morph(dataset_name=dataset_name3,
                            seg_path = seg_data_path,
                            quant_path = quant_data_path, 
                            raw_path = raw_data_path, 
                            raw_file_type = raw_img_type,
                            channel_axis = channel_axis,
                            channel_names = org_file_names,
                            region_names = regions_file_names,
                            mask_name = mask_name,
                            use_scale = use_scale,
                            seg_suffix = suffix_separator)

# compare the output files from both dataset runs to ensure they are identical
morpho_file1 = quant_data_path / f"{dataset_name}_regions_morphology_metrics.csv"
morpho_file3 = quant_data_path / f"{dataset_name3}_regions_morphology_metrics.csv"

# determine if files are identical except for the dataset name column
if morpho_file1.exists() and morpho_file3.exists():
    morph_1 = pd.read_csv(morpho_file1)
    morph_3 = pd.read_csv(morpho_file3)
    morph_3['dataset'] = morph_1['dataset']  # set dataset names to be the same for comparison
    print("\nMorphology metrics files identical:", morph_1.equals(morph_3))

-----
### 🧮 **Summarize metrics *per image/mask* across <INS>ONE OR MORE EXPERIMENTS</ins>**

#### **`STEP 1` - Get the regions morphology .csv files**

#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** Here, the data output from batch_process_regions_morph() from one or more datasets is combined before summarization.

Please specify the list file paths to include in this analysis:
- `csv_path_list`: a list of file paths that include the output csv files from batch_process_regions_morph() for one or more datasets. If there is more than one dataset in each location, they will all be read in here as long as each dataset has a unique name (as specified the by dataset_name variable).

In [ ]:
### USER INPUT REQUIRED ###
csv_path_list = [quant_data_path]

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** These steps collect all of the regions .csv quantification files from the list of locations and combines them so that there is now one data table per analysis type included.

Here, we will just use the single data path specified as quant_data_path earlier in the notebook, but multiple paths can be specified if desired. This path will have a few datasets in it if the above was run through fully.

In [ ]:
# create empty list to hold the regions tables from different experiments
regions_tab = []

# loop through all of the locations listed above and find the _regions_morph files; append them to the list above
for loc in csv_path_list:
    # list all csv files in the location
    files_store = sorted(loc.glob("*.csv"))

    # find the unique datasets in this location based on the prefixes before "_regions_morphology_metrics"
    prefixes = set(f.name.split("_regions_morphology_metrics")[0] for f in files_store if "_regions_morphology_metrics" in f.name)
    for prefix in prefixes:
        # select only the files from this dataset
        files_subset = [f for f in files_store if f.name.startswith(prefix +"_regions_morphology_metrics")]
        for file in files_subset:
            stem = file.stem
            if "_regions_morph" in stem:
                test_regions = pd.read_csv(file, index_col=0)
                regions_tab.append(test_regions)

# combine the regions_morph lists found above into one table
regions_df = pd.concat(regions_tab,axis=0, join='outer').reset_index()

print("Regions metrics from all datasets:")
regions_df

#### **`STEP 2` - Summarize the mean, median, and standard deviation of each metric per image/mask region**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below summarizes the metrics per image/mask. The following are summarized:
- The number, total volume, volume fraction (total org vol/mask volume), and total surface area of each organelle per image/mask
- The mean, median and standard deviation of the "SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", and "axis_major_length" metrics for each organelle per image/mask

In [ ]:
###################
# summary stat group
###################
group_by = ['dataset', 'image_name', 'mask_name', 'scale', 'object']
sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]  + list(regions_df.filter(regex=".*intensity.*").columns)
ag_func_standard = ['mean', 'median', 'std']

###################
# summarize morphology metrics
###################
tab1 = regions_df[group_by + ['label']].groupby(group_by).agg(['count'])
tab1.rename(columns={'label': 'region'}, inplace=True)
tab2 = regions_df[group_by + ['volume', 'surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
tab3 = regions_df[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
regions_summary = pd.merge(tab1, tab2, 'outer', on=group_by)
regions_summary = pd.merge(regions_summary, tab3, 'outer', on=group_by)

# Get mask_name and corresponding volume column per group & calculate volume fraction
mask_names = regions_df.groupby(group_by)['mask_name'].first()
mask_volume_data = regions_df.groupby(group_by).first().apply(lambda row: row[f"{mask_names.loc[row.name]}_volume"], axis=1)
regions_summary.insert(regions_summary.columns.get_loc(('volume', 'sum')) + 1, ('volume', 'fraction'), regions_summary[('volume', 'sum')]/mask_volume_data)

print("Regions morphology summary statistics:")
regions_summary

#### **`STEP 3` - Ensure all regions included in the analysis are represented and fill NA values with 0 as needed**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below adds any regions that were missing from the analysis (no region objects in images) and represents their volumes and counts with 0. If any objects have a count of 1, the standard deviation value is changes to NA.

In [ ]:
# Ensure all possible interactions are represented (if missing fill with NaN)
for ind in regions_summary.index.droplevel(4).unique().to_list():
    for row in regions_file_names:
        if ind+(row,) not in regions_summary.index:
            regions_summary.loc[ind+(row,)] = np.nan

# fill NA with 0 for specific columns
fill_dict = {('region', 'count'): 0, 
            ('volume', 'sum'): 0,
            ('surface_area', 'sum'): 0,
            ('volume', 'fraction'): 0}
regions_summary = regions_summary.fillna(value=fill_dict)

# if (region, count) is 1, set mean, median, and std to NaN
single_site_mask = regions_summary[('region', 'count')] == 1
for col in sharedcolumns+['volume', 'surface_area']:
    regions_summary.loc[single_site_mask, (col, 'std')] = np.nan

regions_summary.sort_index(inplace=True)

print("Regions summary updated:")
display(regions_summary)

#### **`STEP 4` - Unstack the regionsnames and save file**

#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** In this step each table is reformated to "unstack" the organelle column into the rows. In the resulting table, each row corresponds to a single image. The table is then exported.

Please specify if the interactions degree analysis should be carried out:
- `out_prefix`: A string used to name the output files. For example, if "date" is specified as out_prefix, the output files would be named "date_regions_morphology_summarystats.csv".

In [ ]:
### USER INPUT REQUIRED ###
out_prefix = "all_datasets_1"

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below the following action occur:
1. The "organelle" column was unstacked resulting in a dataframe where each column if a different summary statistic and row represents the values for each cell. 
2. For specific values, such as organelle count and total/mean/median volume, NA values were filled with 0.
3. The standard deviation, mean, and median values for the ER (forced to be only one object) were removed.
4. The file was saved.

In [ ]:
# export before unstacking
if (Path(quant_data_path) / f"{out_prefix}_per_region_morphology_summarystats.csv").exists():
    raise FileExistsError(f"CAUTION: {out_prefix}_per_region_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
else:
    regions_summary.to_csv(str(quant_data_path) + f"/{out_prefix}_per_region_morphology_summarystats.csv", mode='x')
    print(f"Exported per-region morphology summary statistics (before unstacking) to {quant_data_path}/{out_prefix}_per_region_morphology_summarystats.csv")

regions_morph_final = regions_summary.unstack(-1)
regions_morph_final.columns = ["_".join((col_name[1], col_name[-1], col_name[0])) for col_name in regions_morph_final.columns.to_flat_index()]
regions_morph_final.columns = [col.replace('sum', 'total') for col in regions_morph_final.columns]
regions_morph_final.columns = [col.replace(col, 'mask_volume') if 'mask' in col else col for col in regions_morph_final.columns]
regions_morph_final = regions_morph_final.loc[:, ~regions_morph_final.columns.duplicated()]
regions_morph_final.reset_index(inplace=True)
print("The regions summary after unstacking:")
display(regions_morph_final)

###################
# export summary sheets
###################
if (Path(quant_data_path) / f"{out_prefix}_regions_morphology_summarystats.csv").exists():
    raise FileExistsError(f"CAUTION: {out_prefix}_regions_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
else:
    regions_morph_final.to_csv(str(quant_data_path) + f"/{out_prefix}_regions_morphology_summarystats.csv", mode='x')
    print(f"Exported regions morphology summary statistics (after unstacking) to {quant_data_path}/{out_prefix}_regions_morphology_summarystats.csv")
print(f"Regions morphology summary is complete.")

#### **`DEFINE` - The batch_regions_morph_summary_stats() function**

In [ ]:
def _batch_regions_morph_summary_stats(csv_path_list: List[str],
                                        out_path: str,
                                        out_prefix: str,
                                        region_names: List[str]):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_prefix: str
        The prefix used to name the output file. An "_" will be included between this prefix and the file suffix.
    region_names: List[str],
        A list of region names used in the region morphology quantification (batch_process_region_morph function)
    """
    # for keeping track of dataset and file numbers
    ds_count = 0
    fl_count = 0

    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    # create empty list to hold the regions tables from different experiments
    regions_tab = []

    # loop through all of the locations listed above and find the _regions_morph files; append them to the list above
    for loc in csv_path_list:
        # list all csv files in the location
        files_store = sorted(loc.glob("*.csv"))

        # find the unique datasets in this location based on the prefixes before "_regions_morphology_metrics"
        prefixes = set(f.name.split("_regions_morphology_metrics")[0] for f in files_store if "_regions_morphology_metrics" in f.name)
        for prefix in prefixes:
            ds_count += 1
            # select only the files from this dataset
            files_subset = [f for f in files_store if f.name.startswith(prefix +"_regions_morphology_metrics")]
            for file in files_subset:
                fl_count += 1
                stem = file.stem
                if "_regions_morph" in stem:
                    test_regions = pd.read_csv(file, index_col=0)
                    regions_tab.append(test_regions)

    # combine the regions_morph lists found above into one table
    regions_df = pd.concat(regions_tab,axis=0, join='outer').reset_index()
    print(f"Found {fl_count} files from {ds_count} dataset(s) across {len(csv_path_list)} location(s).")


    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'mask_name', 'scale', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]  + list(regions_df.filter(regex=".*intensity.*").columns)
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize morphology metrics
    ###################
    tab1 = regions_df[group_by + ['label']].groupby(group_by).agg(['count'])
    tab1.rename(columns={'label': 'region'}, inplace=True)
    tab2 = regions_df[group_by + ['volume', 'surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
    tab3 = regions_df[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
    regions_summary = pd.merge(tab1, tab2, 'outer', on=group_by)
    regions_summary = pd.merge(regions_summary, tab3, 'outer', on=group_by)

    # Get mask_name and corresponding volume column per group & calculate volume fraction
    mask_names = regions_df.groupby(group_by)['mask_name'].first()
    mask_volume_data = regions_df.groupby(group_by).first().apply(lambda row: row[f"{mask_names.loc[row.name]}_volume"], axis=1)
    regions_summary.insert(regions_summary.columns.get_loc(('volume', 'sum')) + 1, ('volume', 'fraction'), regions_summary[('volume', 'sum')]/mask_volume_data)

    #######################
    # fill gaps & NA values
    #######################
    # Ensure all possible interactions are represented (if missing fill with NaN)
    for ind in regions_summary.index.droplevel(4).unique().to_list():
        for row in region_names:
            if ind+(row,) not in regions_summary.index:
                regions_summary.loc[ind+(row,)] = np.nan

    # fill NA with 0 for specific columns
    fill_dict = {('region', 'count'): 0, 
                ('volume', 'sum'): 0,
                ('surface_area', 'sum'): 0,
                ('volume', 'fraction'): 0}
    regions_summary = regions_summary.fillna(value=fill_dict)

    # if (region, count) is 1, set mean, median, and std to NaN
    single_site_mask = regions_summary[('region', 'count')] == 1
    for col in sharedcolumns+['volume', 'surface_area']:
        regions_summary.loc[single_site_mask, (col, 'std')] = np.nan

    regions_summary.sort_index(inplace=True)

    ###################
    # flatten datasheet and export
    ###################
    # export before unstacking
    if (Path(out_path) / f"{out_prefix}_per_region_morphology_summarystats.csv").exists():
        raise FileExistsError(f"CAUTION: {out_prefix}_per_region_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `out_path` to continue without error.")
    else:
        regions_summary.to_csv(str(out_path) + f"/{out_prefix}_per_region_morphology_summarystats.csv", mode='x')
        print(f"Exported per-region morphology summary statistics (before unstacking) to {out_path}/{out_prefix}_per_region_morphology_summarystats.csv")
    regions_morph_final = regions_summary.unstack(-1)
    regions_morph_final.columns = ["_".join((col_name[1], col_name[-1], col_name[0])) for col_name in regions_morph_final.columns.to_flat_index()]
    regions_morph_final.columns = [col.replace('sum', 'total') for col in regions_morph_final.columns]
    regions_morph_final.columns = [col.replace(col, 'mask_volume') if 'mask' in col else col for col in regions_morph_final.columns]
    regions_morph_final = regions_morph_final.loc[:, ~regions_morph_final.columns.duplicated()]
    regions_morph_final.reset_index(inplace=True)

    ###################
    # export summary sheets
    ###################
    if (Path(out_path) / f"{out_prefix}_regions_morphology_summarystats.csv").exists():
        raise FileExistsError(f"CAUTION: {out_prefix}_regions_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `out_path` to continue without error.")
    else:
        regions_morph_final.to_csv(str(out_path) + f"/{out_prefix}_regions_morphology_summarystats.csv", mode='x')
        print(f"Exported regions morphology summary statistics (after unstacking) to {out_path}/{out_prefix}_regions_morphology_summarystats.csv")
    print(f"Regions morphology summary is complete.")
    return regions_summary

In [ ]:
# define new out_prefix for comparison
out_prefix2 = "all_datasets_1-comparison"

# test the function above
test_regions_summary = _batch_regions_morph_summary_stats(csv_path_list = csv_path_list,
                                                            out_path = quant_data_path,
                                                            out_prefix = out_prefix2,
                                                            region_names = regions_file_names)

# compare the output files from both dataset runs to ensure they are identical
morpho_file1 = quant_data_path / f"{out_prefix}_regions_morphology_summarystats.csv"
morpho_file2 = quant_data_path / f"{out_prefix2}_regions_morphology_summarystats.csv"
# determine if files are identical except for the dataset name column
if morpho_file1.exists() and morpho_file2.exists():
    morpho_1 = pd.read_csv(morpho_file1)
    morpho_2 = pd.read_csv(morpho_file2)
    print("\nRegions summarystats files identical:", morpho_1.equals(morpho_2))

##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.regions` and can be imported with the following:
> ```python
> from infer_subc.quantification.regions import batch_regions_morph_summary_stats
> ```

In [ ]:
from infer_subc.quantification.regions import batch_regions_morph_summary_stats

# define new out_prefix for comparison
out_prefix3 = "all_datasets_1-final"

# test the function above
final_regions_summary = batch_regions_morph_summary_stats(csv_path_list = csv_path_list,
                                                            out_path = quant_data_path,
                                                            out_prefix = out_prefix3,
                                                            region_names = regions_file_names)

# compare the output files from both dataset runs to ensure they are identical
morpho_file1 = quant_data_path / f"{out_prefix}_regions_morphology_summarystats.csv"
morpho_file3 = quant_data_path / f"{out_prefix3}_regions_morphology_summarystats.csv"
# determine if files are identical except for the dataset name column
if morpho_file1.exists() and morpho_file3.exists():
    morpho_1 = pd.read_csv(morpho_file1)
    morpho_3 = pd.read_csv(morpho_file3)
    print("\nRegions summarystats files identical:", morpho_1.equals(morpho_3))

-----
-----

## **EXECUTE QUANTIFICATION** <a id='execute'></a>

### **`STEP 1` - 🧪 Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `dataset_name`: A unique string identifier for the dataset being processed. It will be included as metadata in output tables and as part of the output files names. It will be used to identify if any data has already been collected for this dataset.
- `raw_path`: Path or str to the folder that contains the raw image files
- `seg_path`: Path or str to the folder that contains the segmentation tiff files
- `quant_path`: Path or str to the folder that the output datatables will be saved to
- `raw_file_type`: File type of the raw images (e.g., "czi", "tiff")
- `region_names`: List of region names to analyze. Usually ['cell', 'nuc'] for cell mask and nucleus.
    If no regions are to be included, specify None here.
- `channel_axis`: Axis corresponding to the channels in the image data
- `channel_names`: List of channel names associated to each channel in the raw image data; if you wish to exclude a particular channel
    from the intensity analysis, write None instead of the channel name.
- `mask_name`: Name of the region to use for segmentation (if any). This name should be included in the regions_name variable.
    If None, the entire image will be quantified.
- `use_scale`: Whether to apply scaling to the quantitative data; scaled data will be in real world units (e.g., microns) rather than pixels/voxels
- `seg_suffix`: Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix, not including the initial "-"


*The defaults below utilize the user input from the `IMPORTS` section. Update as necessary.*

In [ ]:
dataset_name = "20241204_test"
seg_path = seg_data_path
quant_path = quant_data_path
raw_path = raw_data_path
raw_file_type = raw_img_type
channel_axis = channel_axis
channel_names = org_file_names
region_names = regions_file_names
mask_name = mask_name
use_scale = True
seg_suffix = suffix_separator

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block uses the inputs provided above to run the batch processing. The table that is saved to you files is also printed below for easy access.

In [ ]:
batch_process_regions_morph(dataset_name=dataset_name,
                            seg_path = seg_path,
                            quant_path = quant_path, 
                            raw_path = raw_path, 
                            raw_file_type = raw_file_type,
                            channel_axis = channel_axis,
                            channel_names = channel_names,
                            region_names = region_names,
                            mask_name = mask_name,
                            use_scale = use_scale,
                            seg_suffix = seg_suffix)

### **`STEP 2` - 🧮 Summarize metrics *per cell* across <INS>ONE OR MORE EXPERIMENTS</ins>**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `csv_path_list`: A list of path strings where .csv files to analyze are located.
- `out_path`: A path string where the summary data file will be output to
- `out_preffix`: The prefix used to name the output file. An "_" will be included between this prefix and the file suffix.
- `region_names`: A list of region names used in the region morphology quantification (batch_process_region_morph function)

*The defaults below utilize the user input from the `IMPORTS` section. Update as necessary.*

In [ ]:
csv_path_list = csv_path_list
out_path = quant_data_path
out_prefix = out_prefix
region_names = regions_file_names

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block uses the inputs provided above to run the batch processing. The table that is saved to you files is also printed below for easy access.

In [ ]:
batch_regions_morph_summary_stats(csv_path_list = csv_path_list,
                                  out_path = out_path,
                                  out_prefix = out_prefix,
                                  region_names = region_names)

-----
-----
# 🎉 **CONGRATULATIONS!!**
### **You've successfully quantified organelle morphology using the modular `2.4_cell_regions_morphology` notebook.**

Continue on to other quantification notebooks as needed:
- [2.1 Organelle morphology](2.1_organelle_morphology.ipynb)
- [2.2 Organelle interactions](2.2_organelle_interactions.ipynb)
- [2.3 Subcellular distribution](2.3_organelle_distribution.ipynb)